In [1]:
import glob
import os

base_directory = "./ArtStylesDataset"

# El patrón "**" busca recursivamente en subcarpetas.
# "*.jpg" busca archivos que terminen en .jpg (puedes cambiarlo a *.* para todo)
patron = os.path.join(base_directory, "**", "*.jpg")

# recursive=True es vital para que entre a las carpetas
all_image_paths = glob.glob(patron, recursive=True)

print(f"Total de imágenes encontradas: {len(all_image_paths)}")
all_image_paths[:10] # Muestra las primeras 10 rutas

Total de imágenes encontradas: 487


['./ArtStylesDataset\\anime\\040c478de4f1f64d1d37ebbfda54eac6.jpg',
 './ArtStylesDataset\\anime\\04dda196c55f940108745a086cb6c3c3.jpg',
 './ArtStylesDataset\\anime\\09c43aff05c6cd858a3ea605a8a6aff3.jpg',
 './ArtStylesDataset\\anime\\0ce368c52284b97a84d3b4222e31c71d.jpg',
 './ArtStylesDataset\\anime\\1329f1e905393f7bd9a60af5247a6615.jpg',
 './ArtStylesDataset\\anime\\164bf9489823033a3306a0bd227e3e29.jpg',
 './ArtStylesDataset\\anime\\17e3f90a8e5f532046e39062414448f9.jpg',
 './ArtStylesDataset\\anime\\190d56f7981e941877cdb6753c0235b8.jpg',
 './ArtStylesDataset\\anime\\1b8b5db5f0d14357be7ae3a495e69da4.jpg',
 './ArtStylesDataset\\anime\\1d6997234aea5d195599ce5fdd70b331.jpg']

In [3]:
sample_image_path = all_image_paths[:487]
# Mantener las rutas tal como las devuelve glob; no volver a añadir base_directory
sample_image_path

['./ArtStylesDataset\\anime\\040c478de4f1f64d1d37ebbfda54eac6.jpg',
 './ArtStylesDataset\\anime\\04dda196c55f940108745a086cb6c3c3.jpg',
 './ArtStylesDataset\\anime\\09c43aff05c6cd858a3ea605a8a6aff3.jpg',
 './ArtStylesDataset\\anime\\0ce368c52284b97a84d3b4222e31c71d.jpg',
 './ArtStylesDataset\\anime\\1329f1e905393f7bd9a60af5247a6615.jpg',
 './ArtStylesDataset\\anime\\164bf9489823033a3306a0bd227e3e29.jpg',
 './ArtStylesDataset\\anime\\17e3f90a8e5f532046e39062414448f9.jpg',
 './ArtStylesDataset\\anime\\190d56f7981e941877cdb6753c0235b8.jpg',
 './ArtStylesDataset\\anime\\1b8b5db5f0d14357be7ae3a495e69da4.jpg',
 './ArtStylesDataset\\anime\\1d6997234aea5d195599ce5fdd70b331.jpg',
 './ArtStylesDataset\\anime\\2033948466e8e8a939731b0ef4e52ec5.jpg',
 './ArtStylesDataset\\anime\\24cea8056c2fc6bbdbb6862a05348014.jpg',
 './ArtStylesDataset\\anime\\25c7efa8bfe8db44a79ff630d11cb479.jpg',
 './ArtStylesDataset\\anime\\26f58fea6d1e947fff8fc8674433b040.jpg',
 './ArtStylesDataset\\anime\\2e578f10da5533f99b2

In [4]:
from pandas import DataFrame
from PIL import Image


# Asumimos que 'all_image_paths' es la lista que creamos en el paso anterior
# Nota: DataFrame usa un diccionario o una lista de listas, ajusté ligeramente tu sintaxis
payloads = DataFrame({"image_path": all_image_paths})

# 1. os.path.dirname(x) obtiene la carpeta contenedora: "./ArtStylesDataset/anime"
# 2. os.path.basename(...) obtiene el nombre final de esa ruta: "anime"
payloads["type"] = payloads["image_path"].apply(lambda x: os.path.basename(os.path.dirname(x)))

# Veamos el resultado
print(payloads.head())

# Cuenta cuántas filas hay por cada 'type'
print(payloads["type"].value_counts())

                                          image_path   type
0  ./ArtStylesDataset\anime\040c478de4f1f64d1d37e...  anime
1  ./ArtStylesDataset\anime\04dda196c55f940108745...  anime
2  ./ArtStylesDataset\anime\09c43aff05c6cd858a3ea...  anime
3  ./ArtStylesDataset\anime\0ce368c52284b97a84d3b...  anime
4  ./ArtStylesDataset\anime\1329f1e905393f7bd9a60...  anime
type
anime              75
digital_scenery    74
cubism             58
digital_potrait    56
cartoon            54
surrealism         53
caricature         44
cyberpunk          37
sketch             36
Name: count, dtype: int64


In [5]:
# Crea imágenes PIL a partir de las rutas locales obtenidas anteriormente.

# Usando la columna que ya definimos en el DataFrame anterior
# Limita el lote para evitar agotamiento de memoria y omite archivos defectuosos.
images = []
max_load = 64
for p in payloads['image_path'].tolist()[:max_load]:
    try:
        img = Image.open(os.path.normpath(p)).convert('RGB')
        images.append(img)
    except Exception as e:
        print(f'No se pudo abrir {p}: {e}')
len(images)

64

In [6]:
# Genera representaciones base64 junto con los metadatos para previsualizar sin deformar.
from io import BytesIO
import math
import base64
import os
from PIL import Image

target_width = 256

def resize_image(image_path, target_width=target_width, upscale=False):
    try:
        image_path = os.path.normpath(image_path)
        pil_image = Image.open(image_path).convert('RGB')
    except FileNotFoundError:
        print(f"Archivo no encontrado: {image_path}")
        return None
    except Exception as e:
        print(f"Error abriendo {image_path}: {e}")
        return None

    w, h = pil_image.size
    if w == 0 or h == 0:
        return None

    # No agrandar imágenes más pequeñas por defecto
    if not upscale and w <= target_width:
        return pil_image.copy()

    # Calcular altura preservando relación de aspecto: new_h = target_width * h / w
    new_w = target_width
    new_h = max(1, math.floor(target_width * h / w))
    resized_pil_image = pil_image.resize((new_w, new_h), resample=Image.LANCZOS)
    return resized_pil_image

def convert_image_to_base64(pil_image, fmt="JPEG", quality=85):
    if pil_image is None:
        return None
    image_data = BytesIO()
    pil_image.save(image_data, format=fmt, quality=quality)
    image_data.seek(0)
    base64_string = base64.b64encode(image_data.read()).decode("utf-8")
    return base64_string

resized_images = list(map(resize_image, sample_image_path))
base64_strings = list(map(convert_image_to_base64, resized_images))
# Crear un mapeo path -> base64 sólo para las rutas procesadas correctamente
path_base64 = {p: b for p, b in zip(sample_image_path, base64_strings) if b is not None}
payloads["base64"] = payloads["image_path"].map(path_base64)
payloads

,image_path,type,base64
0,./ArtStylesDataset\anime\040c478de4f1f64d1d37e...,anime,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBA...
1,./ArtStylesDataset\anime\04dda196c55f940108745...,anime,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBA...
2,./ArtStylesDataset\anime\09c43aff05c6cd858a3ea...,anime,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBA...
3,./ArtStylesDataset\anime\0ce368c52284b97a84d3b...,anime,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBA...
4,./ArtStylesDataset\anime\1329f1e905393f7bd9a60...,anime,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBA...
...,...,...,...
482,./ArtStylesDataset\surrealism\f0b42bfbaa41e2f8...,surrealism,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBA...
483,./ArtStylesDataset\surrealism\f36d4855e0737e9d...,surrealism,/9j/4AAQSkZJRgABAQAAAQABAAD//gAfQ29tcHJlc3NlZC...
484,./ArtStylesDataset\surrealism\f6bb8e9e55ed751a...,surrealism,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBA...
485,./ArtStylesDataset\surrealism\f84433d73dce1d76...,surrealism,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBA...


In [8]:
# Carga el procesador y el modelo; después genera los embeddings del lote inicial.
from transformers import AutoImageProcessor, ResNetForImageClassification

processor = AutoImageProcessor.from_pretrained('microsoft/resnet-50')
model = ResNetForImageClassification.from_pretrained('microsoft/resnet-50')

# Asegurarse de que 'images' está definido en este kernel; si no, cargar un pequeño lote seguro
if 'images' not in globals():
    images = []
    max_load = 32
    paths = payloads['image_path'].tolist() if 'payloads' in globals() else all_image_paths[:487]
    for p in paths[:max_load]:
        try:
            images.append(Image.open(os.path.normpath(p)).convert('RGB'))
        except Exception as e:
            print(f'No se pudo abrir {p}: {e}')
    print(f'Cargadas {len(images)} imágenes para procesar')

inputs = processor(images, return_tensors='pt')

outputs = model(**inputs)
embeddings = outputs.logits
embeddings

Loading weights: 100%|██████████| 320/320 [00:01<00:00, 257.14it/s, Materializing param=resnet.encoder.stages.3.layers.2.layer.2.normalization.weight]              


tensor([[ -9.8926,  -9.0152,  -9.0497,  ..., -10.2149,  -8.3366,  -7.7358],
        [-12.0218,  -9.3105,  -9.1333,  ..., -12.3814, -10.2115,  -9.6567],
        [ -8.0500,  -8.3565,  -9.1601,  ...,  -9.3151,  -7.7144,  -6.5318],
        ...,
        [-11.3069,  -9.7126,  -9.0793,  ..., -11.3924,  -8.7073,  -6.6460],
        [ -8.7463,  -6.8071,  -1.6111,  ...,  -9.4062,  -8.9718,  -7.5287],
        [-10.6696, -10.1492,  -8.5622,  ..., -10.1727, -10.2194,  -7.5065]],
       grad_fn=<AddmmBackward0>)

In [9]:
# Conserva la dimensión del embedding para configurar la colección.
embedding_length = len(embeddings[0])
embedding_length

1000

In [10]:
# Carga las variables de entorno desde el archivo .env del mismo directorio.
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
# Inicializa Qdrant con las variables de entorno cargadas.
import os

from qdrant_client import QdrantClient

qclient = QdrantClient(
    url=os.getenv('QDRANT_DB_URL'),
    api_key=os.getenv('QDRANT_API_KEY')
)

qclient

In [12]:
# Crea de nuevo la colección que almacenará los vectores y metadatos.
# Las importaciones alternativas mantienen compatibilidad entre versiones de qdrant-client.
try:
    from qdrant_client.model import VectorParams, Distance
except Exception:
    try:
        from qdrant_client.http import models as qmodels
        VectorParams = qmodels.VectorParams
        Distance = qmodels.Distance
    except Exception:
        from qdrant_client import models as qmodels
        VectorParams = qmodels.VectorParams
        Distance = qmodels.Distance

collection_name = "ArtStyles_images"
collection = qclient.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=embedding_length,
        distance=Distance.COSINE
    )
)

collection

C:\Users\llove\AppData\Local\Temp\ipykernel_5264\2061121791.py:16: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  collection = qclient.recreate_collection(


True

In [21]:
# Procesa y sube todas las imágenes de `sample_image_path` por lotes.
from qdrant_client import QdrantClient, models
import os
from time import sleep
from PIL import Image

# Reinicializa el cliente con las variables de entorno ya cargadas.
qclient = QdrantClient(url=os.getenv('QDRANT_DB_URL'), api_key=os.getenv('QDRANT_API_KEY'))

batch_size = 64
max_retries = 5
uploaded = 0
total = len(sample_image_path)
print(f'Iniciando carga de {total} imágenes en lotes de {batch_size}')

for i in range(0, total, batch_size):
    batch_paths = sample_image_path[i:i+batch_size]
    imgs = []
    payloads_batch = []
    for p in batch_paths:
        try:
            imgs.append(Image.open(os.path.normpath(p)).convert('RGB'))
            row = payloads[payloads['image_path'] == p]
            if len(row):
                payloads_batch.append(row.to_dict(orient='records')[0])
            else:
                payloads_batch.append({'image_path': p})
        except Exception as e:
            print(f'Error abriendo {p}: {e}')
    if not imgs:
        continue
    inputs = processor(imgs, return_tensors='pt')
    outputs = model(**inputs)
    try:
        emb_batch = outputs.logits.detach().cpu().tolist()
    except Exception:
        emb_batch = outputs.logits.tolist()

    points_batch = []
    for j, vec in enumerate(emb_batch):
        point_id = i + j
        points_batch.append(models.PointStruct(id=point_id, payload=payloads_batch[j], vector=vec))

    attempt = 0
    while True:
        try:
            qclient.upsert(collection_name=collection_name, points=points_batch)
            uploaded += len(points_batch)
            print(f'Lote {i//batch_size} cargado ({len(points_batch)} puntos). Total: {uploaded}')
            break
        except Exception as e:
            attempt += 1
            print(f'Error al subir batch {i//batch_size}: {e} (intento {attempt}/{max_retries})')
            if attempt >= max_retries:
                raise
            sleep(2 ** attempt)

# Comprobar conteo final en Qdrant
res = qclient.count(collection_name=collection_name)
print('Puntos en la colección:', res.count)
print('Esperados:', total, 'Cargados:', uploaded)

Iniciando carga de 487 imágenes en lotes de 64
Error al subir batch 0: The write operation timed out (intento 1/5)
Lote 0 cargado (64 puntos). Total: 64
Error al subir batch 1: The write operation timed out (intento 1/5)
Error al subir batch 1: The write operation timed out (intento 2/5)
Error al subir batch 1: The write operation timed out (intento 3/5)
Lote 1 cargado (64 puntos). Total: 128
Error al subir batch 2: The write operation timed out (intento 1/5)
Error al subir batch 2: The write operation timed out (intento 2/5)
Lote 2 cargado (64 puntos). Total: 192
Lote 3 cargado (64 puntos). Total: 256
Error al subir batch 4: The write operation timed out (intento 1/5)
Error al subir batch 4: The write operation timed out (intento 2/5)
Lote 4 cargado (64 puntos). Total: 320
Error al subir batch 5: The write operation timed out (intento 1/5)
Error al subir batch 5: The write operation timed out (intento 2/5)
Error al subir batch 5: The write operation timed out (intento 3/5)
Error al su

In [25]:
# Sincroniza en MySQL los metadatos de todas las imágenes subidas a Qdrant.

import os
import uuid
from pathlib import Path
from PIL import Image, ExifTags
from qdrant_client import QdrantClient

import mysql.connector

DATASET_PREFIX = 'ArtStylesDataset/'
REQUIRED_METADATA_FIELDS = ('title', 'author_name', 'style_name', 'source_name', 'source_url')

def clean_meta(value):
    if value is None:
        return None
    if isinstance(value, bytes):
        for encoding in ('utf-16le', 'utf-8', 'latin-1'):
            text = value.decode(encoding, errors='ignore').replace('\x00', '').strip()
            if text:
                return text
        return None
    if isinstance(value, (list, tuple)):
        parts = [clean_meta(item) for item in value]
        return ', '.join(part for part in parts if part) or None
    text = str(value).strip()
    return text or None

def read_artwork_metadata(image_path):
    with Image.open(image_path) as img:
        exif = {ExifTags.TAGS.get(k, k): v for k, v in (img.getexif() or {}).items()}
        info = img.info or {}

    return {
        'title': clean_meta(exif.get('XPTitle') or exif.get('ImageDescription') or info.get('Title') or info.get('Description')),
        'author_name': clean_meta(exif.get('XPAuthor') or exif.get('Artist') or info.get('Author') or info.get('Artist')),
        'source_name': clean_meta(exif.get('XPSubject') or info.get('Subject')),
        'source_url': clean_meta(exif.get('Copyright') or info.get('Copyright')),
    }

def normalize_file_path(value):
    # Guardar una ruta relativa y estable, independientemente del separador recibido.
    text = str(value).strip().replace('\\', '/')
    while text.startswith('./'):
        text = text[2:]
    marker_index = text.casefold().find(DATASET_PREFIX.casefold())
    if marker_index >= 0:
        text = text[marker_index:]
    text = '/'.join(part for part in text.split('/') if part not in ('', '.'))
    return os.path.normpath(text)

def path_key(value):
    return normalize_file_path(value).replace('\\', '/').casefold()

def load_qdrant_path_map():
    mapping = {}
    offset = None
    while True:
        records, offset = qclient.scroll(
            collection_name=collection_name,
            limit=256,
            offset=offset,
            with_payload=['image_path', 'file_path', 'path'],
            with_vectors=False,
        )
        for record in records:
            payload = record.payload or {}
            payload_path = payload.get('image_path') or payload.get('file_path') or payload.get('path')
            if not payload_path:
                continue
            key = path_key(payload_path)
            point_id = str(record.id)
            if key in mapping and mapping[key] != point_id:
                raise RuntimeError(f'Qdrant contiene mas de un punto para la ruta {payload_path}')
            mapping[key] = point_id
        if offset is None:
            break
    return mapping

def open_mysql_connection():
    return mysql.connector.connect(
        host=os.getenv('MYSQL_HOST', '127.0.0.1'),
        port=int(os.getenv('MYSQL_PORT', '3306')),
        user=os.getenv('MYSQL_USER', 'root'),
        password=os.getenv('MYSQL_PASSWORD', ''),
        database=os.getenv('MYSQL_DATABASE', 'artstylesimilaritysearch_db'),
        charset='utf8mb4',
    )

if 'qclient' not in globals():
    qclient = QdrantClient(url=os.getenv('QDRANT_DB_URL'), api_key=os.getenv('QDRANT_API_KEY'))
if 'collection_name' not in globals():
    collection_name = 'ArtStyles_images'

dataset_images = []
dataset_keys = set()
for image_path in sorted(all_image_paths):
    file_path = normalize_file_path(image_path)
    key = path_key(file_path)
    if key in dataset_keys:
        raise RuntimeError(f'Ruta duplicada en ArtStylesDataset: {file_path}')
    dataset_keys.add(key)
    dataset_images.append((image_path, file_path, key))

qdrant_path_map = load_qdrant_path_map()
missing_in_qdrant = [file_path for _, file_path, key in dataset_images if key not in qdrant_path_map]
if missing_in_qdrant:
    preview = ', '.join(missing_in_qdrant[:10])
    raise RuntimeError(
        f'{len(missing_in_qdrant)} imagenes no tienen un punto asociado en Qdrant. '
        f'Primeras rutas: {preview}'
    )

artwork_rows = []
incomplete_rows = []
for image_path, file_path, key in dataset_images:
    metadata = read_artwork_metadata(os.path.normpath(image_path))
    row = {
        'title': metadata['title'] or '',  # La columna title es NOT NULL; una cadena vacía permite reportarla como faltante.
        'author_name': metadata['author_name'],
        'style_name': Path(os.path.normpath(image_path)).parent.name or None,
        'source_name': metadata['source_name'],
        'source_url': metadata['source_url'],
        'file_path': file_path,
        'id_qdrant_point': qdrant_path_map[key],
        '_path_key': key,
    }
    missing = [field for field in REQUIRED_METADATA_FIELDS if not row.get(field)]
    if missing:
        incomplete_rows.append((file_path, missing))
    artwork_rows.append(row)

mysql_conn = open_mysql_connection()
cursor = mysql_conn.cursor(dictionary=True)
index_created = False
try:
    # La restricción vuelve fiable la identidad por file_path también fuera de esta celda.
    cursor.execute("""
        SELECT COUNT(*) AS total
        FROM information_schema.statistics
        WHERE table_schema = DATABASE()
          AND table_name = 'artworks'
          AND column_name = 'file_path'
          AND non_unique = 0
    """)
    if cursor.fetchone()['total'] == 0:
        cursor.execute("ALTER TABLE artworks ADD UNIQUE INDEX artworks_file_path_unique (file_path)")
        index_created = True
    mysql_conn.commit()

    mysql_conn.start_transaction()
    cursor.execute("""
        SELECT id_artworks, title, author_name, style_name, source_name, source_url,
               file_path, id_qdrant_point
        FROM artworks
    """)
    existing_rows = cursor.fetchall()
    existing_by_key = {}
    for existing in existing_rows:
        key = path_key(existing['file_path'])
        if key in existing_by_key:
            raise RuntimeError(f'Hay rutas equivalentes duplicadas en artworks: {existing["file_path"]}')
        existing_by_key[key] = existing

    stale_rows = [row for key, row in existing_by_key.items() if key not in dataset_keys]
    rows_to_insert = []
    rows_to_update = []
    updated_count = 0
    unchanged_count = 0
    compared_fields = (
        'title', 'author_name', 'style_name', 'source_name', 'source_url',
        'file_path', 'id_qdrant_point'
    )
    for row in artwork_rows:
        existing = existing_by_key.get(row['_path_key'])
        clean_row = {key: value for key, value in row.items() if key != '_path_key'}
        if existing is None:
            rows_to_insert.append(clean_row)
            continue
        clean_row['id_artworks'] = existing['id_artworks']
        rows_to_update.append(clean_row)
        if any(existing.get(field) != clean_row.get(field) for field in compared_fields):
            updated_count += 1
        else:
            unchanged_count += 1

    # Evita choques si dos rutas intercambiaron sus IDs de Qdrant. El cambio no es visible antes del commit.
    sync_token = uuid.uuid4().hex
    cursor.execute(
        "UPDATE artworks SET id_qdrant_point = CONCAT('__sync_', %s, '_', id_artworks)",
        (sync_token,),
    )
    if stale_rows:
        cursor.executemany(
            "DELETE FROM artworks WHERE id_artworks = %s",
            [(row['id_artworks'],) for row in stale_rows],
        )

    if rows_to_update:
        cursor.executemany("""
            UPDATE artworks
            SET title = %(title)s, author_name = %(author_name)s, style_name = %(style_name)s,
                source_name = %(source_name)s, source_url = %(source_url)s,
                file_path = %(file_path)s, id_qdrant_point = %(id_qdrant_point)s
            WHERE id_artworks = %(id_artworks)s
        """, rows_to_update)

    if rows_to_insert:
        cursor.executemany("""
            INSERT INTO artworks
            (title, author_name, style_name, source_name, source_url, file_path, id_qdrant_point)
            VALUES
            (%(title)s, %(author_name)s, %(style_name)s, %(source_name)s, %(source_url)s,
             %(file_path)s, %(id_qdrant_point)s)
        """, rows_to_insert)

    cursor.execute("""
        SELECT COUNT(*) AS total, COUNT(DISTINCT file_path) AS unique_paths,
               COUNT(DISTINCT id_qdrant_point) AS unique_points
        FROM artworks
    """)
    validation = cursor.fetchone()
    expected = len(artwork_rows)
    if (validation['total'], validation['unique_paths'], validation['unique_points']) != (expected, expected, expected):
        raise RuntimeError(f'Validacion final inconsistente: {validation}; esperado: {expected}')

    mysql_conn.commit()
except Exception:
    mysql_conn.rollback()
    raise
finally:
    cursor.close()
    mysql_conn.close()

print('Sincronizacion de artworks completada:')
print(f'  Imagenes encontradas: {len(artwork_rows)}')
print(f'  Insertadas: {len(rows_to_insert)}')
print(f'  Actualizadas: {updated_count}')
print(f'  Sin cambios: {unchanged_count}')
print(f'  Eliminadas: {len(stale_rows)}')
print(f'  Con metadata incompleta: {len(incomplete_rows)}')
print(f'  Indice UNIQUE creado: {index_created}')
for file_path, missing in incomplete_rows[:20]:
    print(f'  INCOMPLETA {file_path}: faltan {", ".join(missing)}')
if len(incomplete_rows) > 20:
    print(f'  ... {len(incomplete_rows) - 20} imagenes incompletas adicionales')

Sincronizacion de artworks completada:
  Imagenes encontradas: 487
  Insertadas: 487
  Actualizadas: 0
  Sin cambios: 0
  Eliminadas: 0
  Con metadata incompleta: 474
  Indice UNIQUE creado: False
  INCOMPLETA ArtStylesDataset\anime\040c478de4f1f64d1d37ebbfda54eac6.jpg: faltan title, author_name, source_name, source_url
  INCOMPLETA ArtStylesDataset\anime\04dda196c55f940108745a086cb6c3c3.jpg: faltan title, author_name, source_name, source_url
  INCOMPLETA ArtStylesDataset\anime\09c43aff05c6cd858a3ea605a8a6aff3.jpg: faltan title, author_name, source_name, source_url
  INCOMPLETA ArtStylesDataset\anime\1329f1e905393f7bd9a60af5247a6615.jpg: faltan title, author_name, source_name, source_url
  INCOMPLETA ArtStylesDataset\anime\164bf9489823033a3306a0bd227e3e29.jpg: faltan title, author_name, source_name, source_url
  INCOMPLETA ArtStylesDataset\anime\17e3f90a8e5f532046e39062414448f9.jpg: faltan title, author_name, source_name, source_url
  INCOMPLETA ArtStylesDataset\anime\1b8b5db5f0d14357b

In [ ]:
# Vaciado manual de artworks. Cambia a True solo para eliminar todos los metadatos.
import os

import mysql.connector

CONFIRM_CLEAR_ARTWORKS = False

if not CONFIRM_CLEAR_ARTWORKS:
    print('Vaciado cancelado: establece CONFIRM_CLEAR_ARTWORKS = True para confirmar.')
else:
    clear_conn = mysql.connector.connect(
        host=os.getenv('MYSQL_HOST', '127.0.0.1'),
        port=int(os.getenv('MYSQL_PORT', '3306')),
        user=os.getenv('MYSQL_USER', 'root'),
        password=os.getenv('MYSQL_PASSWORD', ''),
        database=os.getenv('MYSQL_DATABASE', 'artstylesimilaritysearch_db'),
        charset='utf8mb4',
    )
    clear_cursor = clear_conn.cursor()
    try:
        clear_cursor.execute('TRUNCATE TABLE artworks')
        clear_conn.commit()
        print('Tabla artworks vaciada. El siguiente id_artworks sera 1.')
    except Exception:
        clear_conn.rollback()
        raise
    finally:
        clear_cursor.close()
        clear_conn.close()

Tabla artworks vaciada. El siguiente id_artworks sera 1.
